# Day 2 · LeetCode 刷题笔记

今天记录：**LeetCode 23（合并 K 个升序链表）**、**LeetCode 86（分隔链表）**、**LeetCode 19（删除链表的倒数第 N 个结点）**。

> 按原始笔记顺序整理，保留自己的理解与易错点，只做必要技术纠错。原笔记把 `mergeKLists` 记成了 LeetCode 28，正确题号是 **23**。下面每题的 `class Solution` 分别针对不同题目，单独使用即可。

## LeetCode 23：合并 K 个升序链表

**难点：** 原本以为完成 `mergeKLists` 前需要先重新定义 `__lt__`，也就是小于号。要纠正一点：**只有直接把 `ListNode` 放进堆并需要比较节点时，才可能用到 `__lt__`；使用 `(head.val, i, head)` 元组方案时不需要。**

如果要自己定义链表节点，`__lt__(self, other)` 中的 `other` 表示另一个待比较的节点。

In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

    # 仅演示：直接把 ListNode 放入 heapq 时，可定义节点间的比较规则。
    # 下方 (val, i, node) 的元组入堆写法不依赖这个方法。
    def __lt__(self, other):
        return self.val < other.val

同时要理解 `import heapq` 操作：

- `heappush(pq, (head.val, i, head))` 是**往最小堆 `pq` 放入一个三元组**，三项依次是节点值、链表索引和节点对象；不是“链表的三维”。
- `heapq` 是堆（heap），**不是堆栈（stack）**。每次 `heappop` 取出最小堆元素。
- 针对一个 `lists` 内的多个链表，要用 `enumerate(lists)`，不是 `range(lists)`。
- **`i` 在同一条链表的后续节点入堆时保持不变**；不同链表的 `i` 不同，恰好可以在值相同时辅助比较，从而避免比较 `ListNode`。

下面是我记录的**入堆、取堆、补入下一个节点的核心代码**；为了让 `mergeKLists` 真正返回合并链表，只补了 `dummy`/`tail` 连接操作。

In [ ]:
import heapq

class Solution:
    def mergeKLists(self, lists):
        pq = []
        for i, head in enumerate(lists):
            if head is not None:
                heapq.heappush(pq, (head.val, i, head))

        dummy = ListNode(-1)
        tail = dummy
        while pq:
            val, i, node = heapq.heappop(pq)
            if node.next is not None:
                heapq.heappush(pq, (node.next.val, i, node.next))
            tail.next = node
            tail = tail.next

        tail.next = None
        return dummy.next

**记住：** `while pq:` 代表堆非空就持续处理；`val, i, node = heapq.heappop(pq)` 才是正确的调用；`node.next` 不是 `None` 才能读取 `node.next.val`。由于同一个链表每次最多只在堆里有一个候选节点，因此元组 `(值, i, 节点)` 中的前两项足以决定比较顺序。

## LeetCode 86：分隔链表

首先要明白，每一次一开始写链表：

In [ ]:
dummy1 = ListNode(-1)
dummy2 = ListNode(-1)
p1 = dummy1
p2 = dummy2

运行到最后的时候，一般来说 `p1` 和 `p2` 已经到了各自链表的尾部，但 `dummy1` / `dummy2` **不是冗余**。

- `dummy1` / `dummy2` 一直是两条链表的**虚拟头节点**。
- `p1` / `p2` 才会随着每次接入节点，不断向后移动。
- 分割完链表之后，**要把已经取走的节点和旧链表后续关系断开**，并在最后连接两部分。

原本记下的关键写法是：当 `p.val >= x` 时，把当前节点接到第二条链表；注意别忘了 `p2 = p2.next`。不管走 `if` 还是 `else`，都要先保存 `temp = p.next`，再把 `p.next = None`。

In [ ]:
class Solution:
    def partition(self, head, x):
        dummy1 = ListNode(-1)
        dummy2 = ListNode(-1)
        p1 = dummy1
        p2 = dummy2
        p = head

        while p is not None:
            if p.val >= x:
                p2.next = p
                p2 = p2.next  # 注意不要忘记将 p2 后移
            else:
                p1.next = p
                p1 = p1.next

            temp = p.next    # 先保存旧链表中的下一个节点
            p.next = None   # 已经取走的 p 和旧链表后续节点断开
            p = temp

        p1.next = dummy2.next  # 把两条分割后的链表连接起来
        return dummy1.next

**我的理解：** `p.next = None` 指的是已经被取走的 `p` 与**旧链表中原来的后继节点**断开；`p = temp` 才能继续遍历原链表。`dummy1.next` 和 `dummy2.next` 则保存两条分区链表的起点。

注意题目要求两组内的**原有相对顺序不变**。

## LeetCode 19：删除链表的倒数第 N 个结点

首先要记住**从后往前找倒数第 `k` 个节点**的范式：快慢指针。

先让 `p1` 前进 `k` 步，再让 `p1` 与 `p2` 同步移动，直到 `p1` 为 `None`；此时 `p2` 指向倒数第 `k` 个节点。（约定 `k` 有效。）

In [ ]:
class Solution:
    def findFromEnd(self, head: ListNode, k: int) -> ListNode:
        p1 = head
        for _ in range(k):
            p1 = p1.next
        p2 = head
        while p1 is not None:
            p1 = p1.next
            p2 = p2.next
        return p2

    def removeNthFromEnd(self, head: ListNode, n: int) -> ListNode:
        dummy = ListNode(-1)
        dummy.next = head
        # 删除倒数第 n 个，要先找到倒数第 n+1 个（前驱）
        x = self.findFromEnd(dummy, n + 1)
        x.next = x.next.next
        return dummy.next

然后要记住：

- **删除倒数第 `n` 个元素**：需要找到它的前驱，所以从后往前要找**倒数第 `n+1` 个节点**。配合 `dummy`，删除原头节点也能统一处理。
- 我原来写“在倒数第 `k` 个元素插入，找 `k-1`”不准确：**插在倒数第 `k` 个节点之前**，找倒数第 `k+1` 个节点（前驱）；**插在它之后**，直接找倒数第 `k` 个节点。

核心原因：`x.next = x.next.next` 修改的是**前驱节点**的指针，所以不是先找到待删节点本身。